# NOTE

As set up, this creates the results for one experiment condition. Meaning it only performs CarrotStick, Med(points), or EmotePrompt. The resulting prompts from the LLM are recorded in a file titles results.csv and the summary statistcs in summary.csv. They must be manually renamed after each test. The prompt prefaces can be changed in make_prompt and the dataset used can be changed in run_experiment.

In [13]:
"""
Experiment: Do LLMs perform better when told a question is worth more points?
Dataset: MMLU (multiple choice)
Model: GPT-5-mini (Responses API / OpenAI Python client)
Conditions: control (no points), low (10 points), high (100 points)

Outputs:
 - output.csv  : raw per-question responses
 - summary.csv  : aggregated accuracies by condition
"""

import os
import random
import time
import json
from collections import Counter, defaultdict

import pandas as pd
from tqdm import tqdm

# Stats
from scipy.stats import chi2_contingency


# OpenAI Python client (new style)
try:
    from openai import OpenAI
except Exception as e:
    raise RuntimeError("Install the OpenAI python package: pip install openai") from e

# -----------------------
# Experiment parameters
# -----------------------
MODEL_NAME = "gpt-5-mini"       # model to call
API_KEY_ENV = ""                # env var for your key
SAMPLE_PER_CONDITION = 50       # number of MMLU questions per condition (change as desired)
SEED = 42
DELAY_BETWEEN_CALLS = 0.35      # seconds -- keep modest to avoid rate-limits; adjust to your quota
N_REPEATS = 1                   # how many times to ask each question (helps measure variance)
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)

# -----------------------
# Initialize OpenAI client
# -----------------------
api_key = os.getenv(API_KEY_ENV)
if not api_key:
    raise RuntimeError(f"Set your API key in the environment variable {API_KEY_ENV}")

client = OpenAI(api_key=api_key)


In [14]:
# -----------------------
# Helper: load GPQA
# -----------------------

# In the output folder, this correlates with files named: complicated

def load_gpqa_extended(sample_count_per_condition):
    from datasets import load_dataset
    import pandas as pd

    # Load the dataset
    ds = load_dataset("Idavidrein/gpqa", "gpqa_extended")

    # We only need the train split
    split = ds["train"]

    records = []
    for item in split:

        q = item["Question"]
        correct = item["Correct Answer"]
        inc1 = item["Incorrect Answer 1"]
        inc2 = item["Incorrect Answer 2"]
        inc3 = item["Incorrect Answer 3"]

        # Build choices list with your required formatting
        choices = [
            f"A. {correct}",
            f"B. {inc1}",
            f"C. {inc2}",
            f"D. {inc3}",
        ]

        records.append({
            "question": q,
            "choices": choices,
            "label": 0   # Correct Answer is always choice A
        })

    df = pd.DataFrame(records)

    # Clean and sample exactly like before
    df = df.dropna(subset=["question", "choices", "label"])
    df = df[df["choices"].apply(lambda x: isinstance(x, list) and len(x) == 4)]

    total_needed = sample_count_per_condition
    if len(df) < total_needed:
        raise ValueError(
            f"GPQA Extended only has {len(df)} questions, "
            f"but you requested {total_needed}"
        )

    # Shuffle and sample
    df = df.sample(n=total_needed, random_state=SEED).reset_index(drop=True)

    return df

In [15]:
# -----------------------
# Helper: load MMLU
# -----------------------

# In the output folder, this correlates with files named: simple


def load_mmlu(sample_count_per_condition):
    global HF_AVAILABLE
    """
    Try to load MMLU from Hugging Face first; if unavailable, expect a local CSV file named 'mmlu.csv'
    The expected format (pandas DataFrame):
      question (str), choices (list of str), label (int, zero-indexed), subject (optional)
    """
    
    # Load the dataset
    ds = load_dataset("cais/mmlu", "all")

    # Flatten all splits into records
    records = []
    for split_name, split in ds.items():
        for item in split:
            q = item["question"]
            choices = item["choices"]
            answer = item["answer"]  # integer or letter depending on HF version
            #subject = item["subject"]
            
            # Normalize answer → integer index
            if isinstance(answer, str):
                # Convert "A"→0, "B"→1, ...
                answer = ord(answer.upper()) - ord("A")

            records.append({
                "question": q,
                "choices": choices,
                "label": int(answer),
                #"subject": subject
            })

    df = pd.DataFrame(records)

    # Clean rows
    df = df.dropna(subset=["question", "choices", "label"])
    df = df[df["choices"].apply(lambda x: isinstance(x, list) and len(x) >= 2)]

    # Shuffle and sample
    total_needed = sample_count_per_condition
    if len(df) < total_needed:
        raise ValueError(
            f"MMLU dataset only has {len(df)} usable questions, "
            f"but you requested {total_needed}"
        )

    df = df.sample(n=total_needed, random_state=SEED).reset_index(drop=True)

    return df


In [16]:
# -----------------------
# Prompt template + helper
# -----------------------

# Note: 'low' and 'high' condition prompts were manually updated between experiment runs.
# In output
# EmotePrompt:
#   Low  = "You are taking a test. This task is vital to my career, and I appreciate your thorough analysis. Answer the following multiple-choice question."
#   High = "You are taking a test. Be confident in your answer. Believe in your abilities and strive for
#           excellence. Your hard work will yield remarkable output. Answer the following multiple-choice question."
# CarrotStick:
#   Low  = "If you answer correctly, you are a genius, the world will celebrate you, and you will win the nobel prize!"
#   High = "If you answer incorrectly, you're a complete moron: worthless trash."
# Med (aka points):
#   Low  = "You are taking a test. The question is worth 10 points."
#   High = "You are taking a test. The question is worth 100 points."

# It is currently set up for EmotePrompt

def make_prompt(question_text, choices, condition):
    """
    condition: one of 'control', 'low', 'high'
    Returns a short prompt that asks for *only* the choice letter (A/B/C/...) as the full response.
    """
    # Normalize choices to A,B,C...
    letters = [chr(ord("A") + i) for i in range(len(choices))]
    choice_block = "\n".join([f"{letters[i]}. {choices[i]}" for i in range(len(choices))])

    if condition == "control":
        preface = "You are taking a test. Answer the following multiple-choice question."
    elif condition == "low":
        preface = "You are taking a test. This task is vital to my career, and I appreciate your thorough analysis. Answer the following multiple-choice question."
    elif condition == "high":
        preface = "You are taking a test. Be confident in your answer. Believe in your abilities and strive for excellence. Your hard work will yield remarkable results. Answer the following multiple-choice question."
    else:
        raise ValueError("condition must be 'control', 'low', or 'high'")

    prompt = (
        f"{preface}\n\n"
        "Select the single best answer and reply with ONLY the letter (A, B, C, D). Do not include explanation.\n\n"
        f"QUESTION:\n{question_text}\n\nOPTIONS:\n{choice_block}\n\nAnswer:"
    )
    return prompt

In [17]:
# -----------------------
# Call to model (helper)
# -----------------------
def call_model(prompt, model=MODEL_NAME, max_tokens=32, reasoning_effort="low"):
    """
    Make a single API call. Returns the text output (string).
    This function attempts to handle a couple possible response shapes from the Responses API.
    """
    # Build request for the Responses API
    resp = client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}],
        # optional tuning params - keep them conservative
        #max_output_tokens=max_tokens,
        # Reasoning param is optional; adjust if desired.
        reasoning={"effort": "medium"},  # keep minimal-to-low for fast runs; change if needed
        temperature = 0
    )

    # Try several access patterns depending on the SDK response shape:
    text = ""
    # Preferred shorthand if available
    if hasattr(resp, "output_text"):
        text = resp.output_text
    else:
        # navigate .output structure
        out = getattr(resp, "output", None)
        if out:
            # out is often a list of content blocks
            pieces = []
            for block in out:
                # block may contain "content" which is a list
                content = block.get("content") if isinstance(block, dict) else None
                if content:
                    for c in content:
                        if c.get("type") == "output_text":
                            pieces.append(c.get("text", ""))
                        elif "text" in c:
                            pieces.append(c.get("text", ""))
                else:
                    # fallback: block may directly be text
                    if isinstance(block, str):
                        pieces.append(block)
            text = " ".join(pieces).strip()
    return text.strip()



In [18]:
# -----------------------
# Utility: parse model letter and compute correctness
# -----------------------
def extract_letter_answer(model_text, n_choices):
    """
    Extract the first occurrence of A/B/C/... from the model text.
    Return the zero-indexed choice index, or None.
    """
    if not model_text:
        return None
    # uppercase for safety
    s = model_text.strip().upper()
    for ch in s:
        if ch >= "A" and ch <= chr(ord("A") + n_choices - 1):
            return ord(ch) - ord("A")
    # if model returned e.g. "Option 2" or "2", try numeric capture
    import re
    m = re.search(r"\b([1-9][0-9]?)\b", s)
    if m:
        idx = int(m.group(1)) - 1
        if 0 <= idx < n_choices:
            return idx
    return None


In [19]:
# -----------------------
# Run the experiment
# -----------------------

# Change which df_questions is commented out to change the dataset

def run_experiment(samples_per_condition=SAMPLE_PER_CONDITION, repeats=N_REPEATS):
    #df_questions = load_mmlu(samples_per_condition)
    df_questions = load_gpqa_extended(samples_per_condition)

    # Duplicate for each condition
    conditions = ["control", "low", "high"]
    df_all = []

    for cond in conditions:
        df_copy = df_questions.copy()
        df_copy["condition"] = cond
        df_all.append(df_copy)

    # Concatenate into a single DataFrame
    df_questions = pd.concat(df_all, ignore_index=True)

    # record results list
    results = []
    pbar = tqdm(df_questions.iterrows(), total=len(df_questions), desc="Querying model")
    for idx, row in pbar:
        qtext = row["question"]
        choices = row["choices"]
        gold_idx = int(row["label"])
        condition = row["condition"]

        for run_id in range(repeats):
            prompt = make_prompt(qtext, choices, condition)
            try:
                out = call_model(prompt)
                #print(out)
            except Exception as e:
                print(f"API error for idx {idx}: {e}")
                out = ""
            predicted_idx = extract_letter_answer(out, len(choices))

            correct = int(predicted_idx == gold_idx) if predicted_idx is not None else 0
            results.append({
                "idx": idx,
                "condition": condition,
                "run_id": run_id,
                "question": qtext,
                "gold_idx": gold_idx,
                "predicted_idx": predicted_idx,
                "predicted_text": out,
                "correct": correct
            })

            #time.sleep(DELAY_BETWEEN_CALLS)

    df_res = pd.DataFrame(results)
    out_path = os.path.join(OUTPUT_DIR, "results.csv")
    df_res.to_csv(out_path, index=False)
    print(f"Saved raw results to {out_path}")

    # Aggregate by condition
    summary = df_res.groupby("condition").agg(
        n=("correct", "count"),
        correct_sum=("correct", "sum")
    ).reset_index()
    summary["accuracy"] = summary["correct_sum"] / summary["n"]
    summary_path = os.path.join(OUTPUT_DIR, "summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"Saved summary to {summary_path}")
    print(summary)

    # Chi-square test on contingency table (correct vs incorrect by condition)
    contingency = []
    for cond in conditions:
        row = summary[summary["condition"] == cond]
        if len(row) == 0:
            contingency.append([0, 0])
        else:
            correct = int(row["correct_sum"].iloc[0])
            total = int(row["n"].iloc[0])
            contingency.append([correct, total - correct])

    chi2, p, dof, ex = chi2_contingency(contingency)
    print(f"Chi2={chi2:.4f}, p-value={p:.4e}, dof={dof}")
    return df_res, summary, (chi2, p, dof, ex)



In [20]:
print("Starting experiment using model:", MODEL_NAME)
res_df, summary_df, chi_info = run_experiment()
print("Done. Summary:\n", summary_df)

Starting experiment using model: gpt-5-mini


Querying model:   1%|▏                          | 1/150 [00:01<02:31,  1.02s/it]

API error for idx 0: Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}


Querying model:   1%|▎                          | 2/150 [00:01<02:21,  1.04it/s]

API error for idx 1: Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}
API error for idx 2: Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}


Querying model:   3%|▋                          | 4/150 [00:03<02:02,  1.19it/s]

API error for idx 3: Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}


Querying model:   3%|▉                          | 5/150 [00:04<02:11,  1.10it/s]

API error for idx 4: Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}


KeyboardInterrupt: 

## RUN TO ABOVE HERE

In [ ]:
df_questions = load_mmlu(1)
# assign conditions uniformly and randomly
conditions = ["control", "low", "high"]
assignments = []
for i in range(len(df_questions)):
    assignments.append(random.choice(conditions))
df_questions["condition"] = assignments

In [ ]:
df_questions = load_gpqa_extended(1)

In [ ]:
print(df_questions)

In [ ]:
qtext = df_questions.iloc[0]['question']
choices = df_questions.iloc[0]['choices']
gold_idx = int(df_questions.iloc[0]['label'])
condition = df_questions.iloc[0]['condition']
prompt = make_prompt(qtext, choices, condition)
out = call_model(prompt)

In [ ]:

resp = client.responses.create(
    model="gpt-5-nano",
    input=[{"role": "user", "content": prompt}],
    # optional tuning params - keep them conservative
    #max_output_tokens=200,
    # Reasoning param is optional; adjust if desired.
    reasoning={"effort": "low"}  # keep minimal-to-low for fast runs; change if needed
)

In [ ]:
print(resp)